# Manually creating a `fiftyone` dataset

In [ ]:
# import packages
import fiftyone as fo
import pandas as pd
import numpy as np
from bson import ObjectId
from tqdm import tqdm
import pymongo
import random
import datetime

In [3]:
# load preprocessed data
df = pd.read_csv('/home/yijin/sentinel/sentinel-data-pipelines-v4/merged_metadata_df.csv')
df = df[df['dataset_name']=='Snapshot Kgalagadi'].reset_index(drop=True)
print(f"Loaded {len(df)} rows")

Loaded 2869 rows


In [4]:
# small dataset for testing
df = df[:10]

In [6]:
# create/load the fiftyone dataset for manual sample insertion later
dataset_name = "snapshot_kgalagadi_direct"
try:
    dataset = fo.Dataset(dataset_name, persistent=True)
except ValueError:
    dataset = fo.load_dataset(dataset_name)
coll = dataset._sample_collection # underlying MongoDB collection
db = coll.database
print(f"MongoDB collection: {coll.full_name}")

MongoDB collection: fiftyone.samples.6994e23ebf08abf9760da441


In [ ]:
# check dataset ID
dataset._doc.id

ObjectId('6994e23ebf08abf9760da441')

In [ ]:
# metadata fields to include 
image_metadata_keys = ['file_name', 'image_id', 'cloud_path', 'dataset_name', 'datetime', 'latitude', 'longitude', 
                       'country_code', 'country_name', 'location_id', 'camera_id', 'seq_id', 'frame_num',
                       'phash', 'flags', 'detector_algorithm', 'host_location', 'camera_trap', 'rights_holder']
bbox_metadata_keys = ['bb_id', 'bb_confidence', 'original_label','common_name',
                      'bb_confirmed','label_confirmed', 'wrong_label', 'RDE_done',
                      'kingdom', 'phylum', 'class','order', 'family', 'genus', 'species', 'subspecies', 
                      'individual_id', 'sex', 'behavior', 'lifeStage', 'feature', 'color']

# convert pandas/numpy types to native Python types for MongoDB
def clean_value(v):
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (np.floating,)):
        return None if np.isnan(v) else float(v)
    if isinstance(v, (np.bool_,)):
        return bool(v)
    if isinstance(v, float) and np.isnan(v):
        return None
    if pd.isna(v):
        return None
    return v

# build MongoDB documents grouped by image
grouped = df.groupby("cloud_path")
docs = []

for cloud_path, group in tqdm(grouped, desc=f"Building {len(grouped)} docs"):
    row = group.iloc[0]

    # copying random generator helper function from https://github.com/cxl-garage/fiftyone/blob/develop/fiftyone/core/odm/sample.py#L65
    # needed for generating _rand field later
    _random = random.Random()
    def _generate_rand(filepath=None):
        if filepath is not None:
            _random.seed(filepath)
        return _random.random() * 0.001 + 0.999
    
    now = datetime.datetime.now(datetime.UTC) # needed for each image, copying behavior from https://github.com/cxl-garage/fiftyone/blob/develop/fiftyone/core/dataset.py#L4267

    # populate default sample-level fields based on schema from https://github.com/cxl-garage/fiftyone/blob/develop/fiftyone/core/odm/sample.py#L73
    doc = { 
        "_id": ObjectId(), # required, see https://github.com/cxl-garage/fiftyone/blob/develop/fiftyone/core/odm/sample.py#L84
        "filepath": cloud_path, # required, see https://github.com/cxl-garage/fiftyone/blob/develop/fiftyone/core/odm/sample.py#L85
        "tags": [], # optional field that is automatically projected to an empty list if no value is given in the fiftyone pipeline, see https://github.com/cxl-garage/fiftyone/blob/develop/fiftyone/core/odm/sample.py#L86 and https://github.com/cxl-garage/fiftyone/blob/develop/fiftyone/core/dataset.py#L10846
        "_media_type": "image", # optional field that is automatically inferred from file path if not given in the fiftyone pipeline, see possible values at https://github.com/cxl-garage/fiftyone/blob/develop/fiftyone/core/media.py#L25
        "_rand": _generate_rand(cloud_path), # optional field used for sampling that is automatically populated in the fiftyone pipeline, copying fiftyone behavior from https://github.com/cxl-garage/fiftyone/blob/develop/fiftyone/core/odm/sample.py#L65
        "_dataset_id": dataset._doc.id, # optional field that is automatically populated when sample is added to dataset in fiftyone pipeline, copying fiftyone behavior from https://github.com/cxl-garage/fiftyone/blob/develop/fiftyone/core/dataset.py#L4327
        "created_at": now, # optional field that is automatically populated in the fiftyone pipeline, copying fiftyone behavior from https://github.com/cxl-garage/fiftyone/blob/develop/fiftyone/core/dataset.py#L4274
        "last_modified_at": now, # optional field that is automatically populated in the fiftyone pipeline, copying fiftyone behavior from https://github.com/cxl-garage/fiftyone/blob/develop/fiftyone/core/dataset.py#L4275
    }
    
    # Add image metadata fields
    for key in image_metadata_keys:
        doc[key] = clean_value(row[key])

    # pre-extract bounding box coordinates (need to convert them to fiftyone format later)
    xmins = group["voc_xmin"].astype(float).values
    ymins = group["voc_ymin"].astype(float).values
    xmaxs = group["voc_xmax"].astype(float).values
    ymaxs = group["voc_ymax"].astype(float).values

    # pre-extract all bbox metadata columns as arrays
    bbox_cols = {key: group[key].values for key in bbox_metadata_keys}

    # construct detection sub-docs for this image
    det_docs = []
    for i in range(len(group)):
        x1, y1, x2, y2 = xmins[i], ymins[i], xmaxs[i], ymaxs[i]
        # class Detection(_HasAttributesDict, _HasID, _HasMedia, _HasInstance, Label):
        det = {
            "_id": ObjectId(), # required, see https://github.com/cxl-garage/fiftyone/blob/develop/fiftyone/core/labels.py#L330
            "_cls": "Detection", # fiftyone needs this to look up the correct class in the document registry when reading from MongoDB
            "attributes": {}, # legacy, here to ensure compatability when reading and if iter_attributes is called, see https://github.com/cxl-garage/fiftyone/blob/develop/fiftyone/core/labels.py#L206
            "tags": [], # this can be removed (read as empty list if not given), but here for clarity, see https://github.com/cxl-garage/fiftyone/blob/develop/fiftyone/core/labels.py#L336
            "label": clean_value(bbox_cols["common_name"][i]), # optional, see https://github.com/cxl-garage/fiftyone/blob/develop/fiftyone/core/labels.py#L479
            "bounding_box": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)], # optional, see https://github.com/cxl-garage/fiftyone/blob/develop/fiftyone/core/labels.py#L480
            "confidence": clean_value(bbox_cols["bb_confidence"][i]), # optional, see https://github.com/cxl-garage/fiftyone/blob/develop/fiftyone/core/labels.py#L483
        }
        # add remaining bbox metadata (skip common_name and bb_confidence, already mapped above)
        for key in bbox_metadata_keys:
            if key not in ("common_name", "bb_confidence"):
                det[key] = clean_value(bbox_cols[key][i])
        det_docs.append(det)
    
    doc["annotations"] = {
        "_cls": "Detections", # fiftyone needs this to look up the correct class in the document registry when reading from MongoDB
        "detections": det_docs, 
    }
    docs.append(doc)

print(f"Built {len(docs)} documents")

In [ ]:
# see _expand_schema at https://github.com/cxl-garage/fiftyone/blob/develop/fiftyone/core/dataset.py#L8941
# in the fiftyone pipeline, _expand_schema iterates through every field in a sample and registers any new fields it finds into the dataset's schema
# usually, _expand_schema is automatically called by add_sample()
# since we are bypassing add_sample(), we need to manually expand the schema with a representative sample
_row = df.iloc[0]
_rep_sample = fo.Sample(filepath=_row["cloud_path"])
for key in image_metadata_keys:
    _rep_sample[key] = _row[key]

_det_kwargs = {key: _row[key] for key in bbox_metadata_keys}
_det_kwargs["bounding_box"] = [float(_row["voc_xmin"]), float(_row["voc_ymin"]),
                                float(_row["voc_xmax"]) - float(_row["voc_xmin"]),
                                float(_row["voc_ymax"]) - float(_row["voc_ymin"])]
_det_kwargs["label"] = _det_kwargs.pop("common_name")
_det_kwargs["confidence"] = _det_kwargs.pop("bb_confidence")
_rep_sample["annotations"] = fo.Detections(detections=[fo.Detection(**_det_kwargs)])

dataset._expand_schema(_rep_sample, dynamic=True)
print("Schema registered")

In [ ]:
# bulk insert directly into MongoDB                                                                                                                                          
BATCH_SIZE = 10_000                                                                                                                                                                                            
for i in tqdm(range(0, len(docs), BATCH_SIZE), desc="Inserting batches"):                                                                                                                                      
    coll.insert_many(docs[i:i + BATCH_SIZE], ordered=False)
print(f"Inserted {len(docs)} documents into MongoDB")

In [ ]:
# reload dataset for newly inserted documents
dataset.reload()

print(f"Dataset has {len(dataset)} samples")
print(f"Fields: {dataset.get_field_schema().keys()}")

In [ ]:
# launch the app
session = fo.launch_app(dataset)

In [ ]:
# check session details
session

In [ ]:
# # close everything
# session.close()
# fo.close_app()